# Fracture modelling

Use the sections below in order: Train, Validate, then optionally Test. Mixed augmentation is training-only. Final test evaluation is disabled until explicitly enabled. The stored training output is from an earlier interrupted attempt.


## Train

Configure the model in the setup cell and run training. The best validation-macro-F1 checkpoint is saved under `results/shared_pipeline/`.


In [1]:
from pathlib import Path

import pandas as pd
import torch

from sem_fracture.pipeline import run_experiment, run_prediction

DATA_DIR = Path.cwd()
MODEL_KEY = "resnet18"  # resnet18, deit_tiny, tinyvit_11m, tinyvit_21m
MODE = "finetune"  # head_only or finetune
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

BASELINE_CHECKPOINTS = {
    "resnet18": DATA_DIR / "results" / "resnet18_best.pth",
    "deit_tiny": DATA_DIR / "results" / "deit_tiny" / "checkpoints" / "deit_tiny_best.pth",
    "tinyvit_11m": DATA_DIR / "results" / "tinyvit_11m" / "checkpoints" / "tinyvit_11m_best.pth",
    "tinyvit_21m": DATA_DIR / "results" / "tinyvit_21m" / "checkpoints" / "tinyvit_21m_best.pth",
}

if MODE not in ("head_only", "finetune"):
    raise ValueError(f"Unknown training mode: {MODE}")
if MODEL_KEY not in BASELINE_CHECKPOINTS:
    raise ValueError(f"Unknown model key: {MODEL_KEY}")

INITIAL_CHECKPOINT = BASELINE_CHECKPOINTS[MODEL_KEY] if MODE == "finetune" else None
BACKBONE_LR = 1e-5 if MODE == "finetune" else None
HEAD_LR = 1e-4 if MODE == "finetune" else 1e-3
EPOCHS = 30 if MODEL_KEY == "tinyvit_21m" and MODE == "finetune" else 10
PATIENCE = 4 if MODE == "finetune" else None
GRAD_CLIP_NORM = 1.0 if MODEL_KEY.startswith("tinyvit") and MODE == "finetune" else None
CHECKPOINT_PATH = (
    DATA_DIR / "results" / "shared_pipeline"
    / f"{MODEL_KEY}_{MODE}_best.pth"
)
RUN_FINAL_TEST_EVALUATION = False  # enable only after selecting the checkpoint


In [2]:
result = run_experiment(
    data_dir=DATA_DIR,
    model_key=MODEL_KEY,
    mode=MODE,
    checkpoint_path=CHECKPOINT_PATH,
    device=DEVICE,
    head_lr=HEAD_LR,
    backbone_lr=BACKBONE_LR,
    weight_decay=1e-4,
    epochs=EPOCHS,
    patience=PATIENCE,
    grad_clip_norm=GRAD_CLIP_NORM,
    initial_checkpoint=INITIAL_CHECKPOINT,
    augment_mixed=True,
    seed=42,
    batch_size=16,
)

best = result["best_checkpoint"]
print("Best epoch:", best["epoch"])
print("Best validation macro-F1:", f"{best['val_macro_f1']:.4f}")
print("Checkpoint:", CHECKPOINT_PATH)
print("Training split:", result["split_file"])
print("Class weights:", result["class_weights"])


Epoch 1/30 starting



KeyboardInterrupt



## Validate

Evaluate the selected checkpoint on the original validation split. Validation data is not augmented.


In [ ]:
validation_result = run_prediction(
    data_dir=DATA_DIR,
    model_key=MODEL_KEY,
    checkpoint_path=CHECKPOINT_PATH,
    split="validation",
    split_file="dataset_split.csv",
    device=DEVICE,
    batch_size=16,
)

print(validation_result["metrics"])
pd.DataFrame(validation_result["predictions"])


## Test

After selecting the checkpoint with validation, set `RUN_FINAL_TEST_EVALUATION = True` in the setup cell and run the cell below once. It uses the original `dataset_split.csv`; generated Mixed images belong to training only.


In [ ]:
if RUN_FINAL_TEST_EVALUATION:
    final_test_result = run_prediction(
        data_dir=DATA_DIR,
        model_key=MODEL_KEY,
        checkpoint_path=CHECKPOINT_PATH,
        split="test",
        split_file="dataset_split.csv",
        device=DEVICE,
        batch_size=16,
    )
    print("Final held-out test metrics:", final_test_result["metrics"])
else:
    print("Final test evaluation is disabled.")
